# 02 — Sequence Features

Computes four sequence-derived features for every variant in `data/training_variants.csv`:

| Feature | Source |
|---|---|
| `blosum62_score` | BLOSUM62 substitution matrix (Biopython) |
| `conservation_score` | 1 − normalised Shannon entropy from Pfam PF00042 seed MSA |
| `position_entropy` | Shannon entropy H at the aligned column |
| `pssm_score` | PSSM log-odds score for the mutant AA at the target column |

**MSA source**: Pfam seed alignment for the globin family (PF00042) downloaded from the EBI InterPro API. This gives hundreds of pre-aligned globin sequences across species and is authoritative for conservation analysis.

In [1]:
import io
import json
import math
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from Bio import SeqIO
from Bio.Align import substitution_matrices, PairwiseAligner

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

STANDARD_AAS = list('ACDEFGHIKLMNPQRSTVWY')

# Human mature protein sequences (initiator Met included — ClinVar offset already corrected in nb01)
HBB_SEQ = (
    'MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHL'
    'DNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH'
)
HBA1_SEQ = (
    'MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNA'
    'LSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR'
)
GENE_SEQS = {'HBB': HBB_SEQ, 'HBA1': HBA1_SEQ, 'HBA2': HBA1_SEQ}

BLOSUM62 = substitution_matrices.load('BLOSUM62')
print('BLOSUM62 loaded')
print(f'HBB length: {len(HBB_SEQ)}  HBA1 length: {len(HBA1_SEQ)}')

BLOSUM62 loaded
HBB length: 147  HBA1 length: 142


In [2]:
def get_blosum62(wt_aa1: str, mut_aa1: str) -> int:
    """Return BLOSUM62 score for a substitution. Returns -4 if either AA not in matrix."""
    try:
        return int(BLOSUM62[wt_aa1][mut_aa1])
    except KeyError:
        return -4

## Fetch globin homologs from UniProt

Downloads ~250 reviewed UniProt sequences for the globin superfamily (Pfam PF00042). Uses the UniProt REST API — no Pfam download needed.

In [3]:
FASTA_CACHE = DATA_DIR / 'globin_homologs.fasta'

def fetch_globin_homologs(size: int = 250) -> list:
    """Fetch reviewed UniProt sequences for the globin superfamily (PF00042)."""
    params = {
        'query': '(xref:pfam-PF00042) AND (reviewed:true)',
        'format': 'fasta',
        'size': size,
    }
    print(f'Fetching {size} reviewed globin sequences from UniProt...')
    resp = requests.get(
        'https://rest.uniprot.org/uniprotkb/search',
        params=params,
        headers={'Accept': 'text/plain'},
        timeout=60,
    )
    resp.raise_for_status()
    FASTA_CACHE.write_text(resp.text)
    records = list(SeqIO.parse(io.StringIO(resp.text), 'fasta'))
    print(f'Downloaded {len(records)} sequences → {FASTA_CACHE}')
    return records


if FASTA_CACHE.exists():
    print(f'Using cached homologs: {FASTA_CACHE}')
    homologs = list(SeqIO.parse(str(FASTA_CACHE), 'fasta'))
    print(f'Loaded {len(homologs)} sequences')
else:
    homologs = fetch_globin_homologs(size=250)

Using cached homologs: data/globin_homologs.fasta
Loaded 250 sequences


In [4]:
aligner = PairwiseAligner()
aligner.mode = 'global'
aligner.substitution_matrix = BLOSUM62
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5


def build_position_counts(reference: str, homologs: list, min_identity: float = 0.25) -> dict:
    """
    Align each homolog to `reference` and accumulate per-position AA frequency counts.
    Returns {position (1-indexed): {AA: count}} with a Laplace pseudo-count of 1.
    """
    ref_len = len(reference)
    counts = {pos: {aa: 1 for aa in STANDARD_AAS} for pos in range(1, ref_len + 1)}
    used = 0

    for rec in homologs:
        query = str(rec.seq).upper().replace('U', 'C')
        if len(query) < 30:
            continue
        try:
            aln = aligner.align(reference, query)[0]
        except Exception:
            continue

        ref_aln = aln[0]
        qry_aln = aln[1]
        matches = sum(r == q and r in STANDARD_AAS for r, q in zip(ref_aln, qry_aln))
        non_gap = sum(r != '-' for r in ref_aln)
        if non_gap == 0 or matches / non_gap < min_identity:
            continue

        ref_pos = 0
        for r_char, q_char in zip(ref_aln, qry_aln):
            if r_char != '-':
                ref_pos += 1
            if r_char != '-' and q_char in STANDARD_AAS:
                counts[ref_pos][q_char] += 1
        used += 1

    print(f'  Used {used}/{len(homologs)} homologs (>={min_identity:.0%} identity to reference)')
    return counts


print('Aligning to HBB...')
hbb_counts = build_position_counts(HBB_SEQ, homologs)
print('Aligning to HBA1...')
hba1_counts = build_position_counts(HBA1_SEQ, homologs)
GENE_COUNTS = {'HBB': hbb_counts, 'HBA1': hba1_counts, 'HBA2': hba1_counts}

Aligning to HBB...
  Used 219/250 homologs (>=25% identity to reference)
Aligning to HBA1...
  Used 216/250 homologs (>=25% identity to reference)


## Compute per-position conservation statistics

In [5]:
def position_stats(counts: dict) -> dict:
    """From a {AA: count} dict, compute entropy, conservation, and PSSM log-odds."""
    total = sum(counts.values())
    freqs = {aa: c / total for aa, c in counts.items()}
    background = 1 / 20

    H = -sum(f * math.log2(f) for f in freqs.values() if f > 0)
    conservation = max(0.0, 1.0 - H / math.log2(20))
    pssm = {
        aa: round(math.log2(max(freqs.get(aa, 1e-4), 1e-4) / background), 3)
        for aa in STANDARD_AAS
    }
    return {'entropy': round(H, 4), 'conservation': round(conservation, 4), 'pssm': pssm}


hbb_stats  = {pos: position_stats(hbb_counts[pos])  for pos in hbb_counts}
hba1_stats = {pos: position_stats(hba1_counts[pos]) for pos in hba1_counts}
GENE_STATS = {'HBB': hbb_stats, 'HBA1': hba1_stats, 'HBA2': hba1_stats}

# Sanity: position 98 in HBB (heme pocket) should be highly conserved
print(f"HBB position 98 conservation: {hbb_stats[98]['conservation']:.3f}  (expect >0.8)")
print(f"HBB position  6 conservation: {hbb_stats[6]['conservation']:.3f}")

HBB position 98 conservation: 0.477  (expect >0.8)
HBB position  6 conservation: 0.315


## Compute features for all training variants

In [6]:
df = pd.read_csv(DATA_DIR / 'training_variants.csv')
print(f'Loaded {len(df)} training variants')

rows = []
skipped = 0

for _, row in df.iterrows():
    gene = str(row['gene'])
    pos  = int(row['position'])
    wt1  = str(row['wt_aa_1'])
    mut1 = str(row['mut_aa_1'])

    gene_stats = GENE_STATS.get(gene)
    if gene_stats is None or pos not in gene_stats:
        skipped += 1
        continue

    stats = gene_stats[pos]
    rows.append({
        'variant_id':         row['variant_id'],
        'blosum62_score':     get_blosum62(wt1, mut1),
        'conservation_score': stats['conservation'],
        'position_entropy':   stats['entropy'],
        'pssm_score':         stats['pssm'].get(mut1, 0.0),
    })

seq_df = pd.DataFrame(rows)
print(f'Computed sequence features for {len(seq_df)} variants ({skipped} skipped)')
seq_df.describe()

Loaded 153 training variants
Computed sequence features for 153 variants (0 skipped)


,blosum62_score,conservation_score,position_entropy,pssm_score
count,153.000000,153.000000,153.000000,153.000000
mean,-0.359477,0.465410,2.310461,-1.743856
std,1.489500,0.156337,0.675684,1.966529
min,-3.000000,0.197200,0.907800,-3.579000
25%,-1.000000,0.318900,1.825300,-3.542000
50%,0.000000,0.434200,2.445200,-2.536000
75%,1.000000,0.577700,2.943600,-0.747000
max,3.000000,0.790000,3.469600,4.045000


In [7]:
out_path = DATA_DIR / 'sequence_features.csv'
seq_df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

Saved → data/sequence_features.csv


## Save column stats for reuse in notebook 05 (5 Indian variants)

In [8]:
# Save per-position stats as {gene: {position: {entropy, conservation, pssm}}}
export = {}
for gene, stats_dict in GENE_STATS.items():
    export[gene] = {str(pos): stats for pos, stats in stats_dict.items()}

with open(DATA_DIR / 'msa_col_stats.json', 'w') as f:
    json.dump(export, f)
print('Saved per-position conservation stats → data/msa_col_stats.json')

Saved per-position conservation stats → data/msa_col_stats.json
